# Kaggle GPU Runner for TDR Transmission Line Fault & Load Identification

**EE 305 Electromagnetic Waves Project — Theme 4**  
*Course: EE 305 Electromagnetic Waves, IIT Indore*  
*Project Repository: ML-Based TDR Fault & Load Identification*

---

### ⚠️ Required Kaggle Notebook Settings
- **Accelerator:** GPU (NVIDIA T4 or P100)
- **Internet:** **On** (required to clone repository and install lightweight packages)
- **Persistence:** Files persist in `/kaggle/working/`

This notebook executes the full experimental pipeline (`--mode full`) on a Kaggle GPU:
1. Clones the repository into `/kaggle/working/repo`
2. Installs lightweight dependencies (`h5py`, `pyyaml`, `lightgbm`, `shap`, `pyarrow`, `tabulate`)
3. Executes physics and pipeline tests (`pytest -q tests.py`)
4. Generates full datasets (85,000 samples) and trains all models (Conventional, LightGBM, ResNet1D multi-task, plus ablations and learning curves)
5. Executes evaluation suite E1–E8 and renders all 16 publication-grade figures
6. Compresses all outputs into `/kaggle/working/outputs.zip` for one-click local download


In [ ]:
# 1. Clone Repository (Replace <USERNAME> with your GitHub username)
REPO_URL = "https://github.com/ShlokP06/tdr-fault-load-ml.git"

import os
import shutil
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    print(f"Removing existing {repo_dir} for a clean clone...")
    shutil.rmtree(repo_dir)

!git clone $REPO_URL {repo_dir}
%cd {repo_dir}


In [ ]:
# 2. Install Lightweight Dependencies & Check GPU Environment
!pip install -q h5py pyyaml lightgbm shap pyarrow tabulate

import torch
import numpy as np

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
print("=" * 60)


In [ ]:
# 3. Verify Test Suite (15 physics, pipeline and baseline tests)
!pytest -q tests.py


In [ ]:
# 4. Master Orchestrator: Full Dataset Generation, Training & Evaluation
# Running in full mode on GPU with --skip-existing to allow resume if interrupted
!python run_all.py --mode full --device cuda --skip-existing


In [ ]:
# 5. Package Outputs into outputs.zip and Print Main Comparison
import os
import zipfile
from pathlib import Path

zip_dest = Path("/kaggle/working/outputs.zip")
dirs_to_zip = [
    "results",
    "figures",
    "app/weights",
    "report/figures",
    "report/tables",
]
files_to_zip = [
    "data/dataset_card.md",
]

print(f"Packaging artifacts into {zip_dest}...")
with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in dirs_to_zip:
        p = Path(d)
        if p.exists():
            for f in p.rglob("*"):
                if f.is_file() and not f.name.endswith(".pt_tmp"):
                    rel = f.relative_to(Path("."))
                    zf.write(f, rel)
    for f_str in files_to_zip:
        f = Path(f_str)
        if f.exists():
            zf.write(f, f.relative_to(Path(".")))

zip_size_mb = zip_dest.stat().st_size / (1024 * 1024)
print(f"Outputs successfully zipped! Size: {zip_size_mb:.2f} MB")
print(f"Download path: {zip_dest}")

# Display key comparison table
comparison_md = Path("results/tables/main_comparison.md")
if comparison_md.exists():
    print("\n" + "=" * 60)
    print("FINAL EXPERIMENTAL RESULTS (results/tables/main_comparison.md)")
    print("=" * 60)
    print(comparison_md.read_text(encoding="utf-8"))
